# Swire Coca-Cola Capstone Modeling
### Chris McTeague
### 11/3/2024

# Business Problem Statement

Swire Coca Cola experiences a loss of approximately 60 million dollars a year due to machine down times annually. Currently workers are deployed in a reactive fashion to fix the broken machines but this is a slow process that limits productivity. A solution to this problem would be implementing a predictive model that could alert workers of machines with a high probability of malfunctioning prior to the incident itself thus reducing the time the machine is down and increasing productivity.

In [ ]:
!pip install lifelines


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import statsmodels.api as sm
import missingno as msno
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split, cross_val_score
import pandas as pd # import packages
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from lifelines import KaplanMeierFitter, CoxPHFitter

## Data Cleaning

In [ ]:
#filling in missing 'EQUIP_START_UP_DATE' values that already have a corresponding value based on 'FUNCTIONAL_LOC'
df['EQUIP_START_UP_DATE'] = df.groupby('FUNCTIONAL_LOC')['EQUIP_START_UP_DATE'].transform(lambda group: group.ffill().bfill())
#transforming date columns into datetime datatype
df['EQUIP_START_UP_DATE'] = pd.to_datetime(df['EQUIP_START_UP_DATE'], errors='coerce')
df['EXECUTION_START_DATE'] = pd.to_datetime(df['EXECUTION_START_DATE'])
#creating a year column
df['YEAR'] = df['EXECUTION_START_DATE'].dt.year
#seperating out 'FUNCTIONAL_LOC'
df[['SEGMENT_1', 'SEGMENT_2', 'SEGMENT_3', 'SEGMENT_4', 'SEGMENT_5', 'SEGMENT_6']] = df['FUNCTIONAL_LOC'].str.split('-', expand=True, n=5)
#aranging dataset in ascending order of the below features
df = df.sort_values(by=['FUNCTIONAL_LOC','EXECUTION_START_DATE','ACTUAL_START_TIME'], ascending=[True,True,True])

<ipython-input-79-0ad841e9c39f>:2: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['EQUIP_START_UP_DATE'] = df.groupby('FUNCTIONAL_LOC')['EQUIP_START_UP_DATE'].transform(lambda group: group.ffill().bfill())


In [ ]:
df.head(10)

,ORDER_ID,PLANT_ID,PRODUCTION_LOCATION,EXECUTION_START_DATE,EXECUTION_FINISH_DATE,ACTUAL_START_TIME,ACTUAL_FINISH_TIME,ACTUAL_WORK_IN_MINUTES,MAINTENANCE_PLAN,MAINTENANCE_ITEM,MAINTENANCE_ACTIVITY_TYPE,ORDER_DESCRIPTION,MAINTENANCE_TYPE_DESCRIPTION,FUNCTIONAL_LOC,FUNCTIONAL_AREA_NODE_1_MODIFIED,FUNCTIONAL_AREA_NODE_2_MODIFIED,FUNCTIONAL_AREA_NODE_3_MODIFIED,FUNCTIONAL_AREA_NODE_4_MODIFIED,FUNCTIONAL_AREA_NODE_5_MODIFIED,EQUIPMENT_ID,EQUIPMENT_DESC,EQUIP_CAT_DESC,EQUIP_START_UP_DATE,EQUIP_VALID_FROM,EQUIP_VALID_TO,YEAR,SEGMENT_1,SEGMENT_2,SEGMENT_3,SEGMENT_4,SEGMENT_5,SEGMENT_6
85107,700039229,G221,SUZUKA,2017-05-08,2017-05-08,06:00:00.000,06:00:00.000,15.0,G22121244,90361.0,Planned,STRL01_CONT_MIURA_RUN_2664_99,Preventive Maintenance Order,G221-CLR-A85-E06,COOLER SERVICE,COOLER SERVICE HVAC,COOLER SERVICE HVAC EXT,NaN,NaN,300009603.0,NaN,NaN,NaT,NaN,NaN,2017,G221,CLR,A85,E06,None,None
448477,700039225,G221,SUZUKA,2017-05-08,2017-05-08,06:00:00.000,12:00:00.000,60.0,G22121242,90359.0,Planned,STRL01_MECH_MIURA_CIL_RUN_2662_99,Preventive Maintenance Order,G221-CLR-A85-E06,COOLER SERVICE,COOLER SERVICE HVAC,COOLER SERVICE HVAC EXT,NaN,NaN,300009624.0,NaN,NaN,NaT,NaN,NaN,2017,G221,CLR,A85,E06,None,None
576427,700039260,G221,SUZUKA,2017-05-08,2017-05-08,06:00:00.000,18:00:00.000,45.0,G221VJ5041,90386.0,Planned,BOILER FOR MECH WHILE OPERATING,Preventive Maintenance Order,G221-CLR-A85-E06,COOLER SERVICE,COOLER SERVICE HVAC,COOLER SERVICE HVAC EXT,NaN,NaN,300009603.0,NaN,NaN,NaT,NaN,NaN,2017,G221,CLR,A85,E06,None,None
579926,700039336,G221,SUZUKA,2017-05-08,2017-05-08,06:00:00.000,06:00:00.000,30.0,G22121243,90823.0,Planned,STRL01_MECH_MIURA_CIL_DOWN_2663_99,Preventive Maintenance Order,G221-CLR-A85-E06,COOLER SERVICE,COOLER SERVICE HVAC,COOLER SERVICE HVAC EXT,NaN,NaN,300009624.0,NaN,NaN,NaT,NaN,NaN,2017,G221,CLR,A85,E06,None,None
839930,700039334,G221,SUZUKA,2017-05-08,2017-05-08,06:00:00.000,06:00:00.000,30.0,G22121243,90822.0,Planned,STRL01_MECH_MIURA_CIL_DOWN_2663_99,Preventive Maintenance Order,G221-CLR-A85-E06,COOLER SERVICE,COOLER SERVICE HVAC,COOLER SERVICE HVAC EXT,NaN,NaN,300009603.0,NaN,NaN,NaT,NaN,NaN,2017,G221,CLR,A85,E06,None,None
973591,700039227,G221,SUZUKA,2017-05-08,2017-05-08,06:00:00.000,12:00:00.000,60.0,G22121242,90360.0,Planned,STRL01_MECH_MIURA_CIL_RUN_2662_99,Preventive Maintenance Order,G221-CLR-A85-E06,COOLER SERVICE,COOLER SERVICE HVAC,COOLER SERVICE HVAC EXT,NaN,NaN,300009603.0,NaN,NaN,NaT,NaN,NaN,2017,G221,CLR,A85,E06,None,None
1335342,700039231,G221,SUZUKA,2017-05-08,2017-05-08,06:00:00.000,06:00:00.000,15.0,G22121244,90362.0,Planned,STRL01_CONT_MIURA_RUN_2664_99,Preventive Maintenance Order,G221-CLR-A85-E06,COOLER SERVICE,COOLER SERVICE HVAC,COOLER SERVICE HVAC EXT,NaN,NaN,300009624.0,NaN,NaN,NaT,NaN,NaN,2017,G221,CLR,A85,E06,None,None
134185,700039335,G221,SUZUKA,2017-05-15,2017-05-15,06:00:00.000,06:00:00.000,30.0,G22121243,90822.0,Planned,STRL01_MECH_MIURA_CIL_DOWN_2663_99,Preventive Maintenance Order,G221-CLR-A85-E06,COOLER SERVICE,COOLER SERVICE HVAC,COOLER SERVICE HVAC EXT,NaN,NaN,300009603.0,NaN,NaN,NaT,NaN,NaN,2017,G221,CLR,A85,E06,None,None
401422,700039230,G221,SUZUKA,2017-05-15,2017-05-15,06:00:00.000,06:00:00.000,6.0,G22121244,90361.0,Planned,STRL01_CONT_MIURA_RUN_2664_99,Preventive Maintenance Order,G221-CLR-A85-E06,COOLER SERVICE,COOLER SERVICE HVAC,COOLER SERVICE HVAC EXT,NaN,NaN,300009603.0,NaN,NaN,NaT,NaN,NaN,2017,G221,CLR,A85,E06,None,None
579907,700039232,G221,SUZUKA,2017-05-15,2017-05-15,06:00:00.000,06:00:00.000,6.0,G22121244,90362.0,Planned,STRL01_CONT_MIURA_RUN_2664_99,Preventive Maintenance Order,G221-CLR-A85-E06,COOLER SERVICE,COOLER SERVICE HVAC,COOLER SERVICE HVAC EXT,NaN,NaN,300009624.0,NaN,NaN,NaT,NaN,NaN,2017,G221,CLR,A85,E06,None,None


## Data Processing

Now that our data is properly arranged and we've addressed some of the null values, we can move forward with creating calculated fields that will enhance our model. Additionally, we'll use interpolation to fill in missing values for critical fields, ensuring that our dataset is as complete as possible.

First, we will calculate the 'Time_To_Failure' for each machine, which will serve as our target variable. This calculation will involve determining the time between each breakdown event by looking forward in time.

After computing the 'Time_To_Failure' values, we will address any remaining missing values by filling them with the average time to failure for each respective machine.

In [ ]:
#clean 'ACTUAL_START_TIME' by removing milliseconds (if they exist) and convert to datetime time format
df['ACTUAL_START_TIME'] = pd.to_datetime(df['ACTUAL_START_TIME'].str.split('.').str[0], format='%H:%M:%S').dt.time

#combine 'EXECUTION_START_DATE' and 'ACTUAL_START_TIME' into a single datetime column
df['Maintenance_Start_Datetime'] = pd.to_datetime(df['EXECUTION_START_DATE'].astype(str) + ' ' + df['ACTUAL_START_TIME'].astype(str))

#sort the DataFrame by 'FUNCTIONAL_LOC' and 'Maintenance_Start_Datetime'
df = df.sort_values(by=['FUNCTIONAL_LOC', 'Maintenance_Start_Datetime'], ascending=[True, True])

#create a new column to store the time until the next unplanned maintenance
df['Time_To_Failure'] = None

#loop through each machine group
for loc, group in df.groupby('FUNCTIONAL_LOC'):
    #create a variable to track the next unplanned maintenance date
    next_unplanned_date = None

    #loop over the rows in this group
    for idx in reversed(group.index):
        row = df.loc[idx]

        #if the row represents an "Unplanned" maintenance update next_unplanned_date
        if row['MAINTENANCE_ACTIVITY_TYPE'] == 'Unplanned':
            if next_unplanned_date is not None:
                #calculate the time until the next unplanned maintenance
                time_to_failure = (next_unplanned_date - row['Maintenance_Start_Datetime']).days
                df.at[idx, 'Time_To_Failure'] = time_to_failure
            next_unplanned_date = row['Maintenance_Start_Datetime']
        else:
            #for planned maintenance, calculate the time until the next unplanned maintenance
            if next_unplanned_date is not None:
                time_to_failure = (next_unplanned_date - row['Maintenance_Start_Datetime']).days
                df.at[idx, 'Time_To_Failure'] = time_to_failure

#convert the new column into a integer
df['Time_To_Failure'] = pd.to_numeric(df['Time_To_Failure'], errors='coerce').astype('Int64')

In [ ]:
#filter for missing values
df_filtered = df[df['Time_To_Failure'].isna() == False]
#find the mean of 'Time_To_Failure' for each machine group
overall_mean = df_filtered['Time_To_Failure'].mean()
#round it into an integer
overall_mean = overall_mean.round()
#fill in missing values
df['Time_To_Failure'].fillna(overall_mean, inplace=True)

<ipython-input-82-8b1798d1beb4>:8: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['Time_To_Failure'].fillna(overall_mean, inplace=True)


Next, we will calculate the 'Days_Since_Planned_Maintenance' for each machine. This will involve using the provided date stamps to determine the number of days that have elapsed since each machine received planned maintenance.By calculating this metric, we can gain insights into how maintenance schedules may impact potential failure rates.  

In [ ]:
#create a new column to store the days since last planned maintenance
df['Days_Since_Planned_Maintenance'] = None

#loop through each machine group
for loc, group in df.groupby('FUNCTIONAL_LOC'):
    #track the last planned maintenance date
    last_planned_date = None

    #loop over the rows in this group
    for idx, row in group.iterrows():
        #if the row represents a "Planned" maintenance, update last_planned_date
        if row['MAINTENANCE_ACTIVITY_TYPE'] == 'Planned':
            last_planned_date = row['Maintenance_Start_Datetime']
            df.at[idx, 'Days_Since_Planned_Maintenance'] = 0  #set to 0 on the day of planned maintenance
        else:
            #for non-planned maintenance, calculate days since the last planned maintenance
            if last_planned_date is not None:
                days_since = (row['Maintenance_Start_Datetime'] - last_planned_date).days
                df.at[idx, 'Days_Since_Planned_Maintenance'] = days_since

#convert the new column to integer
df['Days_Since_Planned_Maintenance'] = pd.to_numeric(df['Days_Since_Planned_Maintenance'], errors='coerce').astype('Int64')

In [ ]:
#filter for missing values
df_filtered = df[df['Days_Since_Planned_Maintenance'].isna() == False]
#find the mean of 'Days_Since_Planned_Maintenance' for each machine group
overall_mean = df_filtered['Days_Since_Planned_Maintenance'].mean()
#round it into an integer
overall_mean = overall_mean.round()
#fill in missing values
df['Days_Since_Planned_Maintenance'].fillna(overall_mean, inplace=True)

<ipython-input-84-7e156699be0c>:8: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['Days_Since_Planned_Maintenance'].fillna(overall_mean, inplace=True)


With the 'Time_To_Failure' and 'Days_Since_Planned_Maintenance' fields created, we can proceed to calculate additional metrics that assess the frequency of both unplanned and planned maintenance events over the past month, six months, and year. While these metrics are likely to be highly correlated with one another, we will focus on retaining those that demonstrate the most significant impact on our model's performance. This approach will help streamline our feature set while ensuring we capture the essential influences on machine reliability and failure predictions.

In [ ]:
#create a binary flag for "Unplanned" maintenance
df['Unplanned_Flag'] = (df['MAINTENANCE_ACTIVITY_TYPE'] == 'Unplanned').astype(int)

#convert 'EXECUTION_START_DATE' to datetime if it's not already
df['EXECUTION_START_DATE'] = pd.to_datetime(df['EXECUTION_START_DATE'])

#sort the DataFrame by 'FUNCTIONAL_LOC' and 'EXECUTION_START_DATE'
df = df.sort_values(by=['FUNCTIONAL_LOC', 'EXECUTION_START_DATE'], ascending=[True, True])

#define a rolling window function
def calculate_rolling_unplanned(df, window_size):
    return df.groupby('FUNCTIONAL_LOC')['Unplanned_Flag'].rolling(window=window_size, min_periods=1).sum().reset_index(level=0, drop=True)

#calculate rolling sums for different time windows
days_in_month = 30

df['Unplanned_Rolling_12M'] = calculate_rolling_unplanned(df, window_size=12 * days_in_month)
df['Unplanned_Rolling_6M'] = calculate_rolling_unplanned(df, window_size=6 * days_in_month)
df['Unplanned_Rolling_1M'] = calculate_rolling_unplanned(df, window_size=1 * days_in_month)

In [ ]:
#create a binary flag for "Planned" maintenance
df['Planned_Flag'] = (df['MAINTENANCE_ACTIVITY_TYPE'] == 'Planned').astype(int)

#convert 'EXECUTION_START_DATE' to datetime if it's not already
df['EXECUTION_START_DATE'] = pd.to_datetime(df['EXECUTION_START_DATE'])

#sort the DataFrame by 'FUNCTIONAL_LOC' and 'EXECUTION_START_DATE'
df = df.sort_values(by=['FUNCTIONAL_LOC', 'EXECUTION_START_DATE'], ascending=[True, True])

#define a rolling window function
def calculate_rolling_unplanned(df, window_size):
    return df.groupby('FUNCTIONAL_LOC')['Planned_Flag'].rolling(window=window_size, min_periods=1).sum().reset_index(level=0, drop=True)

#calculate rolling sums for different time windows
days_in_month = 30

df['Planned_Rolling_12M'] = calculate_rolling_unplanned(df, window_size=12 * days_in_month)
df['Planned_Rolling_6M'] = calculate_rolling_unplanned(df, window_size=6 * days_in_month)
df['Planned_Rolling_1M'] = calculate_rolling_unplanned(df, window_size=1 * days_in_month)

## ML Implementation

In [ ]:
# seperate data sets into planned and unplanned
df_unplanned = df[df['MAINTENANCE_ACTIVITY_TYPE']=='Unplanned']
df_planned = df[df['MAINTENANCE_ACTIVITY_TYPE']=='Planned']
df_unplanned = df_unplanned[df_unplanned['Time_To_Failure'] != 0]
df_unplanned.head()

,ORDER_ID,PLANT_ID,PRODUCTION_LOCATION,EXECUTION_START_DATE,EXECUTION_FINISH_DATE,ACTUAL_START_TIME,ACTUAL_FINISH_TIME,ACTUAL_WORK_IN_MINUTES,MAINTENANCE_PLAN,MAINTENANCE_ITEM,MAINTENANCE_ACTIVITY_TYPE,ORDER_DESCRIPTION,MAINTENANCE_TYPE_DESCRIPTION,FUNCTIONAL_LOC,FUNCTIONAL_AREA_NODE_1_MODIFIED,FUNCTIONAL_AREA_NODE_2_MODIFIED,FUNCTIONAL_AREA_NODE_3_MODIFIED,FUNCTIONAL_AREA_NODE_4_MODIFIED,FUNCTIONAL_AREA_NODE_5_MODIFIED,EQUIPMENT_ID,EQUIPMENT_DESC,EQUIP_CAT_DESC,EQUIP_START_UP_DATE,EQUIP_VALID_FROM,EQUIP_VALID_TO,YEAR,SEGMENT_1,SEGMENT_2,SEGMENT_3,SEGMENT_4,SEGMENT_5,SEGMENT_6,Maintenance_Start_Datetime,Time_To_Failure,Days_Since_Planned_Maintenance,Unplanned_Flag,Unplanned_Rolling_12M,Unplanned_Rolling_6M,Unplanned_Rolling_1M,Planned_Flag,Planned_Rolling_12M,Planned_Rolling_6M,Planned_Rolling_1M,Equipment_Occurrence_Flag,Cumulative_Equipment_Replacements,EQUIP_START_UP_DATE_NUM,Machine_Age
1016872,702711887,G221,SUZUKA,2021-01-16,2021-01-16,05:50:04,08:00:00.000,360.0,NaN,NaN,Unplanned,SURFACE BLOW VALVE IS NOT OPERATING,Corrective Maintenance Order,G221-CLR-A85-E06,COOLER SERVICE,COOLER SERVICE HVAC,COOLER SERVICE HVAC EXT,NaN,NaN,300009603.0,NaN,NaN,2018-09-02 08:57:36,NaN,NaN,2021,G221,CLR,A85,E06,None,None,2021-01-16 05:50:04,40,4,1,1.0,1.0,1.0,0,359.0,179.0,29.0,1,791,1.535879e+09,866 days 15:02:24
314204,700108001,G221,SUZUKA,2017-07-25,2017-07-25,07:00:00,07:00:00.000,45.0,NaN,NaN,Unplanned,CLEAN UP STATION,Corrective Maintenance Order,G221-PRD,SUZUKA PRODUCTION,NaN,NaN,NaN,NaN,300009149.0,NaN,NaN,2016-05-05 00:00:00,NaN,NaN,2017,G221,PRD,None,None,None,None,2017-07-25 07:00:00,343,10,1,1.0,1.0,1.0,0,0.0,0.0,0.0,1,1,1.462406e+09,446 days 00:00:00
973849,700702956,G221,SUZUKA,2018-07-03,2018-07-03,07:00:00,08:50:09.000,270.0,NaN,NaN,Unplanned,METAL STORAGE SHED MOVED,Corrective Maintenance Order,G221-PRD,SUZUKA PRODUCTION,NaN,NaN,NaN,NaN,300009149.0,NaN,NaN,2016-05-05 00:00:00,NaN,NaN,2018,G221,PRD,None,None,None,None,2018-07-03 07:00:00,245,146,1,2.0,2.0,2.0,0,1.0,1.0,1.0,1,3,1.462406e+09,789 days 00:00:00
928119,701239235,G221,SUZUKA,2019-03-05,2019-03-05,08:00:00,08:00:00.000,480.0,NaN,NaN,Unplanned,3/5/19,Corrective Maintenance Order,G221-PRD,SUZUKA PRODUCTION,NaN,NaN,NaN,NaN,300009149.0,NaN,NaN,2016-05-05 00:00:00,NaN,NaN,2019,G221,PRD,None,None,None,None,2019-03-05 08:00:00,1,392,1,3.0,3.0,3.0,0,1.0,1.0,1.0,1,4,1.462406e+09,1034 days 00:00:00
314206,701241831,G221,SUZUKA,2019-03-06,2019-03-06,08:00:00,08:00:00.000,480.0,NaN,NaN,Unplanned,3/6/19,Corrective Maintenance Order,G221-PRD,SUZUKA PRODUCTION,NaN,NaN,NaN,NaN,300009149.0,NaN,NaN,2016-05-05 00:00:00,NaN,NaN,2019,G221,PRD,None,None,None,None,2019-03-06 08:00:00,2,393,1,4.0,4.0,4.0,0,1.0,1.0,1.0,1,5,1.462406e+09,1035 days 00:00:00


In [ ]:
df_unplanned['event'] = 1
df_unplanned['time'] = df_unplanned['Time_To_Failure']
kmf = KaplanMeierFitter()

# Create an empty list to store results
results = []

# Group by FUNCTIONAL LOC and EQUIPMENT ID
groups = df_unplanned.groupby(['FUNCTIONAL_LOC','EQUIPMENT_ID'])

for (loc, equip), group in groups:
    # Fit the model
    kmf.fit(group['time'], event_observed=group['event'])

    # Get survival function
    survival_function = kmf.survival_function_

    # Create a DataFrame with results
    for time, prob in survival_function.iterrows():
        results.append({
            'FUNCTIONAL_LOC': loc,
            'EQUIPMENT_ID': equip,
            'time': time,
            'predicted_lifespan': prob.values[0]
        })

# Convert the results list to a DataFrame
equiploc_results_df = pd.DataFrame(results)

# Optionally, pivot the DataFrame if you want to have times as columns
results_df_pivot = equiploc_results_df.pivot_table(index='FUNCTIONAL_LOC',
                                             columns='time',
                                             values='predicted_lifespan').reset_index()

# Display the results
print(equiploc_results_df.head())

     FUNCTIONAL_LOC  EQUIPMENT_ID  time  predicted_lifespan
0  G221-CLR-A85-E06   300009603.0   0.0            1.000000
1  G221-CLR-A85-E06   300009603.0  40.0            0.000000
2          G221-PRD   300009149.0   0.0            1.000000
3          G221-PRD   300009149.0   1.0            0.878049
4          G221-PRD   300009149.0   2.0            0.780488


In [ ]:
c_index = concordance_index(equiploc_results_df['time'], -equiploc_results_df['predicted_lifespan'])
print("Concordance Index:", c_index)

Concordance Index: 0.7783222248258035


In [ ]:
df_unplanned['event'] = 1
df_unplanned['time'] = df_unplanned['Time_To_Failure']
kmf = KaplanMeierFitter()

# Create an empty list to store results
results = []

# Group by FUNCTIONAL LOC and EQUIPMENT ID
groups = df_unplanned.groupby('FUNCTIONAL_LOC')

for (loc), group in groups:
    # Fit the model
    kmf.fit(group['time'], event_observed=group['event'])

    # Get survival function
    survival_function = kmf.survival_function_

    # Create a DataFrame with results
    for time, prob in survival_function.iterrows():
        results.append({
            'FUNCTIONAL_LOC': loc,
            'time': time,
            'predicted_lifespan': prob.values[0]
        })

# Convert the results list to a DataFrame
functloc_results_df = pd.DataFrame(results)

# Optionally, pivot the DataFrame if you want to have times as columns
results_df_pivot = functloc_results_df.pivot_table(index='FUNCTIONAL_LOC',
                                             columns='time',
                                             values='predicted_lifespan').reset_index()

# Display the results
print(functloc_results_df.head())

     FUNCTIONAL_LOC  time  predicted_lifespan
0  G221-CLR-A85-E06   0.0            1.000000
1  G221-CLR-A85-E06  40.0            0.000000
2          G221-PRD   0.0            1.000000
3          G221-PRD   1.0            0.883721
4          G221-PRD   2.0            0.790698
